# FORMATION-BASELINE-GATE-001 — Colab T4 gate

**Purpose:** spend ~2–3 hours on one T4 to decide whether another expensive FORMATION/TIE-ROLE mechanism campaign is worth running. This does **not** change the completed S5 NULL verdicts.

The run is adaptive: two 3000-update full-mixture controls first; then either a third fresh full-mixture replication or an identity-only disambiguation probe. State/checkpoints are saved in Google Drive and the run exact-resumes after a disconnect.

**Run:** choose **Runtime → Change runtime type → T4 GPU**, then **Run all**. When complete, upload `FORMATION_BASELINE_GATE_001_RESULTS.zip` back to ChatGPT. Keep the checkpoint ZIP in Drive.


In [ ]:
import sys, json, hashlib, shutil, subprocess
from pathlib import Path

REMOTE='https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH='cymek-next-core-architecture'
EXECUTABLE_COMMIT='9989b8c778f3c3250b93e05d38ac26203358991f'
SCIENCE_COMMIT='c15ad8beb409537db42d075684ea54847a074ebd'
OPERATOR_BLOB='82476951698c1b8754868b955121294c6af6f1b8'
PREREG_BLOB='fd5c0b616dfa24f907a2dca1e0473886cd1cfce4'
REPO=Path('/content/An-Ra-the-new-AGI-baseline-gate')
OP_COPY=Path('/content/formation_baseline_gate_001_colab_v1.py')
PRE_COPY=Path('/content/FORMATION_BASELINE_GATE_001_PREREGISTRATION.json')

if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXECUTABLE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==EXECUTABLE_COMMIT

operator_rel='tools/formation_baseline_gate_001_colab_v1.py'
prereg_rel='docs/cymek/experiments/FORMATION-BASELINE-GATE-001/PREREGISTRATION.json'
got_op=subprocess.check_output(['git','-C',str(REPO),'hash-object',operator_rel],text=True).strip()
got_pre=subprocess.check_output(['git','-C',str(REPO),'hash-object',prereg_rel],text=True).strip()
assert got_op==OPERATOR_BLOB,(got_op,OPERATOR_BLOB)
assert got_pre==PREREG_BLOB,(got_pre,PREREG_BLOB)
shutil.copy2(REPO/operator_rel,OP_COPY); shutil.copy2(REPO/prereg_rel,PRE_COPY)

# Execute against the exact frozen S5 science tree, not later branch code.
subprocess.run(['git','-C',str(REPO),'checkout','-q',SCIENCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SCIENCE_COMMIT

subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime -> Change runtime type -> T4 GPU, then rerun.')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))

from google.colab import drive
drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/CYMEK/FORMATION_BASELINE_GATE_001')
OUT.mkdir(parents=True,exist_ok=True)
binding={'schema':'anra.formation-baseline-gate-colab-binding/v1','executable_commit':EXECUTABLE_COMMIT,'science_commit':SCIENCE_COMMIT,'operator_blob':OPERATOR_BLOB,'prereg_blob':PREREG_BLOB}
bp=OUT/'COLAB_EXECUTABLE_BINDING.json'
if bp.exists(): assert json.loads(bp.read_text())==binding,'Existing Drive state belongs to a different executable; use a new output folder.'
else: bp.write_text(json.dumps(binding,indent=2)+'\n')
print('READY | exact S5 science tree:',SCIENCE_COMMIT)
print('Drive output:',OUT)


In [ ]:
cmd=[sys.executable,'-u',str(OP_COPY),'--repo',str(REPO),'--out',str(OUT),'--prereg',str(PRE_COPY)]
print('Starting/resuming FORMATION-BASELINE-GATE-001...',flush=True)
print('Expected wall time on a Colab T4: roughly 2–3 hours; exact time depends on the assigned GPU and whether Stage B runs full-mixture or identity-only.',flush=True)
proc=subprocess.run(cmd,cwd=REPO)
print('RETURN CODE:',proc.returncode)
if proc.returncode!=0:
    raise SystemExit('Gate stopped/failed. Do NOT delete the Drive folder. Reconnect a T4 and Run all again; compatible checkpoints exact-resume.')
result=json.loads((OUT/'BASELINE_GATE_RESULT.json').read_text())
print('\nDECISION:',result['decision'])
print('EXPENSIVE MECHANISM CAMPAIGN WORTH RUNNING:',result['expensive_mechanism_campaign_worth_running'])
print('REASON:',result['reason'])


In [ ]:
from google.colab import files
result_path=OUT/'BASELINE_GATE_RESULT.json'
results_zip=OUT/'FORMATION_BASELINE_GATE_001_RESULTS.zip'
checkpoints_zip=OUT/'FORMATION_BASELINE_GATE_001_CHECKPOINTS.zip'
if not result_path.exists() or not results_zip.exists() or not checkpoints_zip.exists():
    raise RuntimeError('Experiment is not complete yet. Preserve Drive state and resume Cell 2.')
result=json.loads(result_path.read_text())
print('FINAL:',result['decision'])
print('Worth expensive mechanism run:',result['expensive_mechanism_campaign_worth_running'])
print('Results SHA256:',hashlib.sha256(results_zip.read_bytes()).hexdigest())
print('Checkpoint archive is preserved in Drive:',checkpoints_zip)
print('Downloading the small RESULTS bundle now. Upload this ZIP back to ChatGPT.')
files.download(str(results_zip))
